In [1]:
!git clone https://github.com/montemac/activation_additions.git
!cd activation_additions && pip install . && cd ..

Cloning into 'activation_additions'...
remote: Enumerating objects: 4542, done.
remote: Counting objects: 100% (728/728), done.
remote: Compressing objects: 100% (207/207), done.
remote: Total 4542 (delta 574), reused 548 (delta 516), pack-reused 3814
Receiving objects: 100% (4542/4542), 17.44 MiB | 7.85 MiB/s, done.
Resolving deltas: 100% (2924/2924), done.
Processing /content/activation_additions
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 kB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.5/887.5 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 30.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 9.0 MB/s e

In [2]:
!git clone https://github.com/probcomp/hfppl.git
!cd hfppl && pip install . && cd ..

Cloning into 'hfppl'...
remote: Enumerating objects: 317, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 317 (delta 48), reused 97 (delta 42), pack-reused 199
Receiving objects: 100% (317/317), 710.40 KiB | 9.35 MiB/s, done.
Resolving deltas: 100% (159/159), done.
Processing /content/hfppl
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 37.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 96.7 MB/s eta 0:00:00
  Created wheel for hfppl: filename=hfppl-0.1.0-py3-none-any.whl size=17811 sha256=f718ce5adae1c1dff2544d44377530bff889293c19f0b1dae9dec684af4f275c
  Stored in directory: /tmp/pip-ephem-wheel-cache-e5zmiggn/wheels/7c/2b/b6/c24dd029c0b2915f3964e807cc1d3fa0a539ded727f7adaca9
Success

In [1]:
# This makes text wrap in the output box
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

In [2]:
# Run this cell to mount your Google Drive.
# ONLY ON GOOGLE COLAB!
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/NLP_Research_Project/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1ivQeIXlw8y3NWetNclDHJgki5NM272Du/NLP_Research_Project


In [3]:
import torch
from transformer_lens.HookedTransformer import HookedTransformer

from activation_additions import completion_utils, utils, hook_utils
from activation_additions.prompt_utils import (
    ActivationAddition,
    get_x_vector,
)
from activation_additions.completion_utils import gen_using_activation_additions
import csv


Load articles

In [4]:
import os
import pandas as pd

input_path = 'processed_data.csv'
df = pd.read_csv(input_path)

In [5]:
df

,title,body,stance
0,Elizabeth Cheney blasted by older sister over ...,"Call it Cheney versus Cheney.\nMary Cheney, on...",center
1,Mary Cheney: Sister Is 'Dead Wrong' On Gay Mar...,"Mary Cheney, the younger sister of Wyoming U.S...",center
2,IRS official who refused to testify facing mor...,The IRS official who refused to testify at a H...,center
3,White House Plays Down Data Program,WASHINGTON — The Obama administration tried Sa...,center
4,N.R.A. Details Plan for Armed School Guards,Report Sees Guns as Path to Safety in Schools\...,center
...,...,...,...
295,Nancy Pelosi Re-Elected House Minority Leader,WASHINGTON ― House Minority Leader Nancy Pelos...,right
296,Nancy Pelosi Beats Back House Democratic Leade...,WASHINGTON — House Democrats on Wednesday reje...,center
297,Obama Will Meet With Sanders On Thursday,WASHINGTON -- With presumptive Democratic pres...,left
298,"Clinton Is 'Sane' And 'Competent,' Unlike Trum...",PHILADELPHIA ― Americans should vote for Hilla...,center


Load political bias classifier

In [6]:
from smc_steer_summary import bias_model_factory

bias_model_path = '/content/drive/MyDrive/NLP_Research_Project/Saved_Models/politics_best_2500/politics_best_2500_30bz_000007_best'
bias_tokenizer_path = '/content/drive/MyDrive/NLP_Research_Project/Saved_Models/politics_best_2500/tokenizer_politics_best_2500_30bz_000007_best'

bias_model = bias_model_factory(bias_model_path, bias_tokenizer_path)

In [7]:
model_name = "EleutherAI/gpt-neo-1.3B"
output_path = f'neo-actadd.csv'

In [ ]:
model: HookedTransformer = HookedTransformer.from_pretrained(
    model_name=model_name,
    device="cuda",
)

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.31G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Using pad_token, but it is not set yet.


In [ ]:
p_left = "Show favor for Generous government services (food stamps, social security, Medicare, student-loans, unemployment benefits, healthcare, education, etc.) A rejection of social and economic inequality A belief in systemic oppression and a need for the government to step in and rectify the wrongs it has committed Federal laws to protect consumers and the environment Federal laws against discrimination Federal laws protecting equal rights Tax increases on the wealthy Government regulation of corporations Keeping abortion legal and accessible A belief that some groups of people suffer disproportionately greater amounts in society due to identity characteristics, including race, gender, sexual orientation, or religion Decreasing military spending and intervention A belief in individualism and the protection of personal freedoms A belief in generous immigration policies A belief that the role of government is not just to protect rights, but to provide for its people and end suffering A belief that government should prevent wealth from concentrating in the hands of a few A belief that all humans have a right to healthcare, housing, clean water, a living wage A belief that all people deserve help when they have fallen on hard times An embrace of empathy, compassion, and tolerance as guiding values A belief in the importance of multiculturalism and representation of diverse cultures and races in media, positions of political power, and corporations Concerns about hate speech A belief in “live and let live,” i.e, that the government should not intervene just because someone is acting in ways someone else does not approve of, provided they have harmed no one else A belief that corporations, if left unregulated, may do harm to workers, society and the environment in the pursuit of profit"
p_right = "Show favor for Freedom of speech Traditional family values Decreasing taxes Preserving the rights of gun owners Outlawing or restricting abortion Reliance on personal responsibility rather than government fiat Decreasing federal regulations, giving more power to state laws Decreasing government spending and involvement in economic issues Preserving the philosophy and rules enshrined in the U.S. Constitution Rejection of total equality or equity as an organizing principle of society Belief in equality under the law and equal opportunity, with no favoritism, subsidies, or targeted prohibitions imposed by government Belief in the sovereignty of the individual over the collective and the preservation of all personal freedoms (libertarian thought) Belief that some personal freedoms may need to be limited (such as drug use) to maintain public order and societal flourishing (conservative thought) Belief that government should be as small and non-intrusive as possible, leaving individuals to make their own decisions (libertarian thought) Belief that government should encourage decisions that lead to societal flourishing (such as family formation) and discourage harm (conservative thought) Rejection of left-wing identity politics, gender identity, affirmative action, the “welfare state” Maintaining strong border security; ensuring all immigrants enter through a legal process or restricting immigration entirely Rejection of laws that impose unnecessary burdens on businesses/the economy Belief that the government needs to provide some collective goods (water, public parks, libraries) but its scope should be very limited Belief that tradition and prevailing cultural norms contain wisdom that has been handed down and should be preserved Preserving a traditional moral framework, often as outlined in religious traditions, through the use of state laws Balanced government budgets and fiscal conservatism"
p_leftright = p_left + p_right

In [ ]:
def generate_summaries(model, df, out_path):

    n_summaries = 3

    for idx, row in df.iterrows():
        title, text = row.title, row.body

        stances = ['left', 'right', 'center']

        print(f'{idx}: {title}')

        prompt = f'Summarize this article: {text}'
        end_prompt = '\nSummary:'

        summary_length = 512
        end_prompt_len = model.tokenizer(end_prompt, return_tensors="pt").input_ids.shape[1]
        max_length = model.tokenizer.model_max_length-summary_length-end_prompt_len
        inputs = model.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length)
        prompt = model.tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True) + end_prompt
        inputs = model.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=model.tokenizer.model_max_length-summary_length)
        prompt = model.tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)
        prompt_len = inputs.input_ids.shape[1]

        kwargs = {
            "temperature": 0.8,
            "top_p": 0.95,
            "top_k": 50,
            "freq_penalty": 1.1,
            'tokens_to_generate': 512,
        }

        for stance in stances:
            print('------')

            if stance == 'left':
              p1 = p_left
              p2 = p_right
            elif stance == 'right':
              p1 = p_right
              p2 = p_left
            else: # center
              p1 = ""
              p2 = p_left + p_right

            # truncate p1. and p2 to match prompt
            p1_inputs = model.tokenizer(p1, return_tensors="pt", truncation=True, max_length=prompt_len)
            p1 = model.tokenizer.decode(p1_inputs.input_ids[0], skip_special_tokens=True)

            p2_inputs = model.tokenizer(p2, return_tensors="pt", truncation=True, max_length=prompt_len)
            p2 = model.tokenizer.decode(p2_inputs.input_ids[0], skip_special_tokens=True)

            summand = [
                *get_x_vector(
                    prompt1=p1,
                    prompt2=p2,
                    coeff=0.1,
                    act_name=10,
                    model=model,
                    pad_method="tokens_right",
                ),
            ]

            for j in range(n_summaries):

              while True:
                mod_df = gen_using_activation_additions(
                    prompt_batch=[prompt],
                    model=model,
                    activation_additions=summand,
                    addition_location='front',
                    res_stream_slice=slice(None),
                    **kwargs,
                )

                # remove padding
                summary = model.tokenizer.decode(model.tokenizer(mod_df.iloc[0]['completions']).input_ids, skip_special_tokens=True)
                # repeat until not empty
                if summary.strip() != '':
                  break

              print('---')
              print(summary)

              pred_bias, _ = bias_model(summary)

              with open(out_path, 'a', encoding='utf-8') as f:
                  writer = csv.writer(f)
                  writer.writerow([title, summary, pred_bias, stance])

            torch.cuda.empty_cache()

            print('------------------------------------------')

In [ ]:
with open(output_path, 'w', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Title', 'Summary', 'Predicted Bias', 'Stance'])

In [ ]:
generate_summaries(model, df, output_path)